# Application mart EDA (book AMEX sample)

**Exploration only** — not a production pipeline. Ticket [#147](https://github.com/JLaborda/SmartWealthAI/issues/147).

**Disclaimer:** public demo data only (not live bank BFSI). Source is the *Financial AI in Practice* ch.5 book sample (`credit/data/train_df_sample.pkl`), AMEX-shaped — not Home Credit.

This notebook reads the **application mart** artifact (not the raw pickle as the primary path).

## Generate the mart first

```bash
git lfs pull --include="credit/data/*"
poetry run credit-css build-application-mart \
  --source credit/data/train_df_sample.pkl \
  --output-dir data/credit/application_mart \
  --application-id-column customer_ID \
  --target-column target \
  --bad-value 1 \
  --good-value 0
```

## Target → bad / good

| Column | Bad | Good |
| --- | --- | --- |
| `target` | `1` (default) | `0` (non-default) |

AMEX dictionary: default = no payment within 120 days after the latest statement in the performance window.

In [ ]:
from pathlib import Path

import pandas as pd

MART_DIR = Path("../../data/credit/application_mart")
# Fallback when the notebook cwd is the repo root
if not (MART_DIR / "application_mart.parquet").exists():
    MART_DIR = Path("data/credit/application_mart")

mart = pd.read_parquet(MART_DIR / "application_mart.parquet")
readme = (MART_DIR / "README.md").read_text(encoding="utf-8")
print("shape", mart.shape)
print("columns sample", list(mart.columns[:8]), "...")
print("id unique", mart["customer_ID"].is_unique)
print("--- sidecar (head) ---")
print("\n".join(readme.splitlines()[:25]))

## Class balance (bad / good)

In [ ]:
TARGET = "target"
BAD, GOOD = 1, 0

counts = mart[TARGET].value_counts(dropna=False).sort_index()
rate = (mart[TARGET] == BAD).mean()
balance = pd.DataFrame(
    {
        "label": ["good (0)", "bad (1)"],
        "n": [int((mart[TARGET] == GOOD).sum()), int((mart[TARGET] == BAD).sum())],
    }
)
balance["share"] = balance["n"] / balance["n"].sum()
print(balance.to_string(index=False))
print(f"default rate (bad): {rate:.4%}")
ax = counts.plot(kind="bar", title="target balance (0=good, 1=bad)", rot=0)
ax.set_xlabel("target")
ax.set_ylabel("count")

## Missingness

In [ ]:
miss = mart.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print(f"columns with any NA: {len(miss)} / {mart.shape[1]}")
print(miss.head(20).to_string())
if len(miss):
    miss.head(20).plot(kind="barh", title="Top missingness share", figsize=(8, 6))

## Basic associations (modeling motivation)

Point-biserial style: correlation of numeric features with binary `target` (absolute value), top features. Not WOE/IV — that lands in scoring (#145).

In [ ]:
numeric = mart.select_dtypes(include="number").drop(columns=[TARGET], errors="ignore")
# Sample columns if extremely wide (book sample ~900 features)
corr = numeric.corrwith(mart[TARGET]).abs().sort_values(ascending=False)
top = corr.head(15)
print(top.to_string())
top.plot(kind="barh", title="|corr| with target (top 15)", figsize=(8, 5))

## Feature snapshot vs target

Compare distribution of the strongest numeric association by class.

In [ ]:
feat = corr.index[0]
print("feature", feat, "|corr|", float(corr.iloc[0]))
ax = mart.boxplot(column=feat, by=TARGET, grid=False)
ax.set_title(f"{feat} by target")
ax.set_xlabel("target (0=good, 1=bad)")
ax.figure.suptitle("")

## Notes

- Mart grain: one `customer_ID` per row.
- Full Kaggle AMEX + Home Credit dumps are out of this PR.
- Hermetic CI uses `tests/credit/fixtures/application_source/applications.csv`, not this notebook.